# Diabetes Model Training - PyCaret Pipeline
## PIMA Indians Diabetes Dataset
**Target AUC:** ≥ 0.85  
**Strategy:** AutoML with class imbalance handling

In [ ]:
# Cell 1 — Install dependencies
!pip install pycaret[full] shap optuna --quiet

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MODELS_PATH = '/content/drive/MyDrive/healthai/models/'
import os
os.makedirs(MODELS_PATH, exist_ok=True)

In [ ]:
# Cell 3 — Verify installations
import pycaret
import shap
print(f'PyCaret  {pycaret.__version__}')
print(f'SHAP     {shap.__version__}')
print('Environment ready ✓')

In [ ]:
# Cell 4 — Load Dataset
import pandas as pd

# Download from: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
df = pd.read_csv('diabetes.csv')
print(f'Dataset shape: {df.shape}')
print(f'Class distribution:\n{df["Outcome"].value_counts(normalize=True)}')
df.head()

In [ ]:
# Cell 5 — PyCaret Setup
from pycaret.classification import *

exp = setup(
    data=df,
    target='Outcome',
    session_id=42,
    train_size=0.8,
    fold=5,
    normalize=True,
    fix_imbalance=True,  # Handle class imbalance
    verbose=False
)

print('Setup complete ✓')

In [ ]:
# Cell 6 — Compare Models (AutoML)
best = compare_models(sort='AUC', n_select=1)
print('\nBest model selected:')
print(best)

# View full leaderboard
leaderboard = pull()
print('\nTop 5 models:')
print(leaderboard.head())

In [ ]:
# Cell 7 — Tune Hyperparameters
tuned = tune_model(
    best,
    optimize='AUC',
    n_iter=50,
    search_library='optuna'
)

print('\nTuned model performance:')
print(pull())

In [ ]:
# Cell 8 — Evaluate Model
evaluate_model(tuned)

In [ ]:
# Cell 9 — Finalize & Save Model
final = finalize_model(tuned)
save_model(final, MODELS_PATH + 'diabetes_model')
print(f'✓ Diabetes model saved to {MODELS_PATH}diabetes_model.pkl')

In [ ]:
# Cell 10 — Test Prediction
test_patient = df.iloc[[0]].drop('Outcome', axis=1)
prediction = predict_model(final, data=test_patient)
print('Test prediction:')
print(prediction[['prediction_label', 'prediction_score']])

In [ ]:
# Cell 11 — SHAP Explainer Setup
import shap

# Extract base estimator from pipeline
base_model = final.steps[-1][1]

try:
    explainer = shap.TreeExplainer(base_model)
    print('✓ TreeExplainer created')
except:
    explainer = shap.LinearExplainer(base_model, X_train)
    print('✓ LinearExplainer created')

# Test SHAP values
X_test_sample = get_config('X_test').iloc[[0]]
shap_values = explainer(X_test_sample)
print(f'SHAP values shape: {shap_values.values.shape}')

In [ ]:
# Cell 12 — SHAP Waterfall Plot
shap.plots.waterfall(shap_values[0], max_display=10)

## ✅ Training Complete
- Model saved to Google Drive
- SHAP explainer verified
- Ready for deployment